In [1]:
import brightway2 as bw
import pandas as pd
import openpyxl 
import os
import glob

In [2]:
bw.projects.set_current('Prospective LCA v5')

In [3]:
databaseNames = bw.databases
myDatabaseNames = []
for databaseName in databaseNames:
    if 'Abhi' not in databaseName and 'image' in databaseName:
        myDatabaseNames.append(databaseName)
myDatabaseNames.sort()

abhiDatabaseNames = []
for databaseName in databaseNames:
    if 'Abhi' in databaseName and 'image' in databaseName:
        abhiDatabaseNames.append(databaseName)
abhiDatabaseNames.sort()

In [4]:
myDatabaseNames

['ecoinvent 3.8 cutoff image SSP2-Base 2020',
 'ecoinvent 3.8 cutoff image SSP2-Base 2030',
 'ecoinvent 3.8 cutoff image SSP2-Base 2040',
 'ecoinvent 3.8 cutoff image SSP2-Base 2050',
 'ecoinvent 3.8 cutoff image SSP2-RCP19 2030',
 'ecoinvent 3.8 cutoff image SSP2-RCP19 2040',
 'ecoinvent 3.8 cutoff image SSP2-RCP19 2050',
 'ecoinvent 3.8 cutoff image SSP2-RCP26 2030',
 'ecoinvent 3.8 cutoff image SSP2-RCP26 2040',
 'ecoinvent 3.8 cutoff image SSP2-RCP26 2050']

In [5]:
GWPMethod = [('EF v3.0 no LT', 'ozone depletion no LT', 'ozone depletion potential (ODP)  no LT')]

In [6]:
def breakdown_calculations(db, activity):
    activitiesList = [{activity : 1}]
    for exchange in activity.technosphere():
        activitiesList.append({bw.Database(exchange.input.key[0]).get(exchange.input.key[1]) : exchange.amount})
    calculationSetup = {'inv' : activitiesList, 'ia' : GWPMethod}
    bw.calculation_setups['breakdown'] = calculationSetup
    myLCA = bw.MultiLCA('breakdown')
    results = pd.DataFrame(myLCA.results.transpose(), columns = [str(list(i.keys())[0]).split('\'')[1] for i in activitiesList], index = pd.MultiIndex.from_tuples(GWPMethod))
    results = results.sort_index().drop(index = [i for i in results.index if i[0] == 'rest'])
    directEmissions = pd.DataFrame([0 if abs(results.iloc[r,0] - results.iloc[r,1:].sum()) / abs(results.iloc[r,0]) < 1e-5 else results.iloc[r, 0] - results.iloc[r, 1:].sum() for r in range(len(results.index))],
                                      columns = ['direct emissions'],
                                      index = results.index)
    results = pd.concat([results, directEmissions], axis = 1)
    results['database'] = db.name
    return results

In [7]:
result_dataframes = {
    'heat' : pd.DataFrame(),
    'heatCCS' : pd.DataFrame()
}

In [8]:
for i in range(0, len(myDatabaseNames)):
    
    myDB = bw.Database(myDatabaseNames[i])
    myActivities = list(myDB)

    abhiDB = bw.Database(abhiDatabaseNames[i])
    abhiActivities = list(abhiDB)

    heatActivity = next(filter(
        lambda x: 'heat production, natural gas, at industrial furnace >100kW' in x['name']
                    and 'heat, district or industrial, natural gas' in x['reference product']
                    and 'RoW' in x['location'],
        myActivities
    ))

    heatCCSActivity = next(filter(
        lambda x: 'natural gas heating, with CCS' in x['name']
                    and 'heating, Abhi' in x['reference product']
                    and 'GLO' in x['location'],
        abhiActivities
    ))

    result_dataframes['heat'] = pd.concat(
        [result_dataframes['heat'], breakdown_calculations(myDB, heatActivity)],
        ignore_index = True
    )

    result_dataframes['heatCCS'] = pd.concat(
        [result_dataframes['heatCCS'], breakdown_calculations(abhiDB, heatCCSActivity)],
        ignore_index = True
    )

In [9]:
for result_dataframe in result_dataframes:
   df = result_dataframes[result_dataframe]
   df['name'] = df.columns[0]
   df.set_index('name')
   last_column = df.pop(df.columns[-1])
   df.insert(0, last_column.name, last_column)
   database_column = df.pop(df.columns[-1])
   df.insert(1, database_column.name, database_column)

In [10]:
for result_dataframe in result_dataframes:
   df = result_dataframes[result_dataframe]
   df['database'] = df['database'].str.replace('Base', 'RCP6')
   df['database'] = df['database'].str.replace('PkBudg500', 'RCP19')
   df['database'] = df['database'].str.replace('PkBudg1150', 'RCP26')

In [11]:
heatResultsFileName = 'heat ozone depletion breakdown results.xlsx'

In [12]:
def check_or_create_excel_file(filePath):
    if not os.path.exists(filePath):
        wb = openpyxl.Workbook()
        wb.save(filePath)
        print(f"Excel file created at {filePath}")

def write_dataframes_to_excel(dictionary, file_name):
    check_or_create_excel_file(file_name)
    with pd.ExcelWriter(file_name, engine = 'openpyxl', mode = 'a') as writer:        
        for sheet_name, dataframe in dictionary.items():
            if sheet_name in writer.book.sheetnames:
                writer.book.remove(writer.book[sheet_name])
            dataframe.to_excel(writer, sheet_name = sheet_name)
    wb = openpyxl.load_workbook(file_name)
    if wb.sheetnames[0] == 'Sheet':  # check if the workbook has any sheets
        firstSheet = wb.sheetnames[0]  # get the name of the first sheet
        wb.remove(wb[firstSheet])  # remove the first sheet
        wb.save(file_name)

In [13]:
heat_dataframes = {}

for key in list(result_dataframes.keys()):
        heat_dataframes[key] = result_dataframes[key]

write_dataframes_to_excel(heat_dataframes, os.path.join('..', 'Results', 'Phase 2', heatResultsFileName))

Excel file created at ..\Results\Phase 2\heat ozone depletion breakdown results.xlsx
